# M5 – chạy thí nghiệm đầy đủ (Kaggle hoặc Colab)
Chạy `scripts/run_experiment.py` trên mẫu 1.200 chuỗi. Đặt `SPLIT`, `MODELS`, `BRANCH` ở ô đầu rồi chạy tuần tự.
Kaggle: thêm dữ liệu cuộc thi *M5 Forecasting – Accuracy* vào Input và bật Internet.
Colab: đặt `DATA_DIR` tới thư mục Drive chứa 5 CSV gốc. Không dán token vào notebook.

In [ ]:
import json, os, pathlib, shutil, subprocess, sys
SPLIT = "test"      # "validation" hoặc "test" (test cần tag config-freeze)
MODELS = "catboost"
BRANCH = "config-freeze"   # Test phải chạy đúng mã đã khóa (tag); nhánh m5-end-to-end đã đổi sau khóa
REPO = "https://github.com/haiphamt/IS403.R11-M5-Competition.git"
ON_KAGGLE = os.path.exists("/kaggle/working")
WORK = pathlib.Path("/kaggle/working" if ON_KAGGLE else "/content")
DATA_DIR = None if ON_KAGGLE else "/content/drive/MyDrive/M5"   # Kaggle: bộ đọc tự tìm trong /kaggle/input
OUT = WORK / "runs"   # kết quả và checkpoint nằm NGOÀI thư mục repo nên chạy lại ô clone không làm mất run dở
RUN_ID = None        # đặt run_id cũ (tên thư mục trong OUT) để chạy tiếp một run bị ngắt

In [ ]:
if not ON_KAGGLE:
    from google.colab import drive
    drive.mount("/content/drive")
os.chdir(WORK)  # tránh đứng trong thư mục repo sắp bị xóa
if (WORK / "repo").exists():
    shutil.rmtree(WORK / "repo")  # luôn clone mới để khớp nhánh/tag
subprocess.run(["git", "clone", "--branch", BRANCH, REPO, str(WORK / "repo")], check=True)
os.chdir(WORK / "repo")
print(subprocess.run(["git", "log", "-1", "--format=%H %s"], capture_output=True, text=True).stdout)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "statsforecast", "catboost", "lightgbm", "xgboost", "scikit-learn", "pyarrow", "PyYAML"], check=True)
# So phiên bản thực cài với requirements.txt (chỉ cảnh báo): dự báo XGBoost đổi theo phiên bản (3.2.0 so với 3.4.1)
from importlib.metadata import PackageNotFoundError, version
for _line in open("requirements.txt"):
    _name, _, _pin = _line.strip().partition("==")
    if _pin and _name in ("numpy", "pandas", "pyarrow", "scipy", "scikit-learn", "statsforecast", "numba", "lightgbm", "xgboost", "catboost"):
        try: _got = version(_name)
        except PackageNotFoundError: _got = None
        if _got != _pin: print(f"CẢNH BÁO: {_name} cài {_got}, requirements.txt ghim {_pin}; kết quả có thể khác bản đã lưu")

In [ ]:
# Băm 5 CSV gốc tại thư mục dữ liệu thực sự đọc (runner ở tag cũ ghi data_sha256 rỗng trên Kaggle)
import hashlib
_dirs = [pathlib.Path(DATA_DIR)] if DATA_DIR else sorted({p.parent for p in pathlib.Path("/kaggle/input").rglob("calendar.csv")})
assert len(_dirs) == 1, f"không xác định được duy nhất thư mục dữ liệu: {_dirs}"
OUT.mkdir(parents=True, exist_ok=True)
data_hashes = {f: hashlib.sha256((_dirs[0] / f).read_bytes()).hexdigest() for f in
               ("calendar.csv", "sell_prices.csv", "sales_train_validation.csv", "sales_train_evaluation.csv", "sample_submission.csv")}
(OUT / "data_sha256.json").write_text(json.dumps({"data_dir": str(_dirs[0]), "sha256": data_hashes}, indent=1))
print(data_hashes)
import multiprocessing
cpus = multiprocessing.cpu_count(); print("CPU:", cpus)
cmd = [sys.executable, "scripts/run_experiment.py", "--split", SPLIT, "--models", *MODELS.split(),
       "--workers", str(max(1, cpus - 1)), "--n-jobs", str(cpus)]
cmd += ["--out", str(OUT)]
if RUN_ID: cmd += ["--run-id", RUN_ID]
if DATA_DIR: cmd += ["--data-dir", DATA_DIR]
subprocess.run(cmd, check=True)

In [ ]:
import shutil, glob, pandas as pd
run = max(glob.glob(f"{OUT}/{SPLIT[:3]}-*"), key=os.path.getmtime)  # run vừa tạo của split này (OUT chỉ chứa run của notebook)
print(run); print(pd.read_csv(run + "/metrics_summary.csv").round(4).to_string())
shutil.rmtree(run + "/ckpt", ignore_errors=True)
shutil.copy(OUT / "data_sha256.json", run)  # bằng chứng dữ liệu đi cùng run
print(shutil.make_archive(str(WORK / f"m5_{SPLIT}_{os.path.basename(run)}"), "zip", run))